# 🏆 Workshop: Bau deinen eigenen WM-Experten 2026
## Notebook 1: Verbindung zum MCP-Server & Tool-Aufrufe

In diesem Notebook lernen wir das **Model Context Protocol (MCP)** kennen. MCP ist ein offenes Protokoll, das es KIs (wie ChatGPT, Claude oder euren eigenen Agenten) ermöglicht, auf externe Datenquellen und Tools zuzugreifen.

Unser WM-Experten-Agent benötigt Zugriff auf aktuelle Spieldaten, ELO-Ratings und Simulations-Algorithmen. Diese werden von unserem maßgeschneiderten **MCP-Server** bereitgestellt.

### Lernziele:
1. Verstehen, wie ein MCP-Client mit einem MCP-Server über HTTP kommuniziert.
2. Abfragen der verfügbaren Tools des MCP-Servers.
3. Ausführen von Tools wie ELO-Abfragen und der Turnier-Simulation über Python.


### Schritt 1: Verbindung testen und Tools auflisten
Wir nutzen die offizielle Python-Bibliothek `mcp` und den `streamablehttp_client` Transport, um uns mit dem MCP-Server zu verbinden und alle verfügbaren Tools abzufragen. Da Jupyter-Notebooks eine asynchrone Event-Schleife bereitstellen, können wir `await` direkt im Code verwenden.


In [ ]:
%pip install -q mcp

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
import json

# Die URL des MCP-Servers
MCP_URL = "https://wm2026.fh-swf.cloud/mcp"

try:
    async with streamablehttp_client(MCP_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tools_result = await session.list_tools()
            tools = tools_result.tools
            
            print(f"✅ Erfolgreich verbunden! {len(tools)} Tools gefunden:\n")
            for tool in tools:
                print(f"🛠️  Tool-Name: {tool.name}")
                print(f"   Beschreibung: {tool.description}")
                print("   Parameter:")
                print(json.dumps(tool.inputSchema, indent=4, ensure_ascii=False))
                print("-" * 60)
except Exception as e:
    print(f"❌ Fehler bei der Verbindung zum MCP-Server: {e}")
    print("HINWEIS: Bitte stelle sicher, dass der MCP-Server läuft!")


### Schritt 2: ELO-Statistiken abfragen
Wir rufen nun das Tool `get_team_elo` auf. Dazu nutzen wir die Methode `call_tool` der MCP-Session mit dem gewünschten Tool-Namen und den Parametern.


In [ ]:
async def call_tool(name, parameters={}):
    """Hilfsfunktion, um ein MCP-Tool aufzurufen."""
    async with streamablehttp_client(MCP_URL) as (read_stream, write_stream, _):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            result = await session.call_tool(name, arguments=parameters)
            if not result.content:
                return {}
            try:
                return json.loads(result.content[0].text)
            except Exception:
                return result.content[0].text

# ELO-Daten für Deutschland abfragen
deutschland_elo = await call_tool("get_team_elo", {"team": "Deutschland"})
print(json.dumps(deutschland_elo, indent=2, ensure_ascii=False))


### Schritt 3: Turniersimulation ausführen
Der MCP-Server verfügt über eine Monte-Carlo-Simulation für das gesamte Turnier (`simulate_tournament`). Wir rufen das Tool auf und lassen die Chancen berechnen.


In [ ]:
# Simulation starten
sim_result = await call_tool("simulate_tournament", {"team": "Deutschland", "n_sims": 1000})

print("=== WM 2026 Prognose für Deutschland ===")
print(sim_result.get("summary"))
print("\nVisualisierung der Wahrscheinlichkeiten pro Runde:")
print(sim_result.get("bar_chart"))


### 🧠 Aufgabe für dich:
Schreibe ein kleines Skript, das die Siegwahrscheinlichkeiten der Top-10-Teams abfragt und ausgibt. Nutze dazu das Tool `simulate_tournament` ohne Angabe eines Teams.


In [ ]:
# DEINE LÖSUNG HIER:
top10_result = 
print(top10_result.get("bar_chart"))
